# Step 3 — Real-Time ACT Inference on the Lebai Arm

This notebook loads the policy from `act_training.ipynb` and uses it to drive your Lebai 6-DOF arm + gripper in real time.

## What you'll learn

1. **The inference loop**: how observations get assembled, how predicted action chunks get consumed, and how temporal ensembling smooths the result.
2. **Real Lebai SDK control**: `start_sys()`, `init_claw()`, `movej()` (non-blocking), `set_claw()`, `stop_sys()` — matching the pattern used in your `grasp_to_the_bowl.py`.
3. **Latency budgeting**: at 10 Hz you have 100 ms per tick. Camera grab + policy forward + SDK call + sleep all live in that budget.
4. **Safety practices** that are non-optional when running a learned policy on real hardware.

## Before running this notebook

- A trained checkpoint at `./checkpoints/act_run01/final/`.
- The Lebai arm in a **safe pose**, away from people and fragile things.
- The e-stop within reach.
- The same camera service running at the same URL/IP that you used for data collection. The model was trained on data from that camera; using a different one fails silently in confusing ways.
- Velocity factor on the Lebai pendant turned **down** for your first runs.


## How ACT inference actually works

The training notebook taught the policy to predict the next `chunk_size` actions from one observation. Two things make this work in practice:

**1. Action chunk execution.** Each policy call returns `chunk_size` actions. You execute some prefix and then re-query. If you execute all of them, you've got 3.2 s of "open loop" between updates — too long for anything dynamic. If you execute only one, you waste compute. A good default is to execute about a quarter to a half.

**2. Temporal ensembling.** At every timestep `t`, multiple recent chunks proposed an action *for that timestep `t`* (because chunks overlap). Averaging them with an exponential weight (recent chunks weighted higher) gives a much smoother trajectory than the raw chunk actions, and noticeably improves task success.

LeRobot's `ACTPolicy.select_action()` handles temporal ensembling internally. We'll use that path — call it once per tick, get one ensembled action back, send it to the robot.


## Safety checklist — go through this *out loud* before running the control loop

1. The robot is in a clear workspace. **No person within sweep range.**
2. The e-stop is within arm's reach of whoever's watching.
3. Velocity factor on the pendant is **low** for the first run.
4. You have a way to **stop the cell** (Jupyter "Interrupt Kernel") that does not require unplugging the robot.
5. The robot is in a **starting pose similar to one of your training episodes' first frames**. Generalizing to wildly different starts is exactly what imitation learning fails at.


## 1. Imports

In [ ]:
import base64
import time
from pathlib import Path

import cv2
import numpy as np
import requests
import torch

import lebai_sdk
from lerobot.common.policies.act.modeling_act import ACTPolicy


## 2. Load the trained policy

In [ ]:
CHECKPOINT_PATH = Path("./checkpoints/act_run01/final")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy = ACTPolicy.from_pretrained(CHECKPOINT_PATH)
policy.to(device); policy.eval()
print(f"Loaded policy from {CHECKPOINT_PATH}")
print(f"  chunk_size: {policy.config.chunk_size}")
print(f"  n_action_steps: {policy.config.n_action_steps}")
print(f"  expected inputs:")
for name, spec in policy.config.input_features.items():
    print(f"    {name}: shape={spec.shape}")
print(f"  outputs:")
for name, spec in policy.config.output_features.items():
    print(f"    {name}: shape={spec.shape}")


## 3. Connect to the camera service and the robot

The camera setup mirrors `save_state_img.py`. The robot setup mirrors `grasp_to_the_bowl.py`: connect, then `start_sys()` to enable motion, then `init_claw()` to enable the gripper.


In [ ]:
ROBOT_IP = "192.168.31.254"
CAMERA_URL = "http://192.168.31.192:8000"

class CameraClient:
    def __init__(self, url=CAMERA_URL):
        self.url = url.rstrip('/'); self.s = requests.Session()
    def start_all(self):
        self.s.post(f"{self.url}/cameras/start_all", json={
            "enable_color": True, "enable_depth": False,
            "color_config": {"width": 640, "height": 480, "fps": 30},
        })
    def stop_all(self): self.s.post(f"{self.url}/cameras/stop_all")
    def list_cameras(self): return self.s.get(f"{self.url}/cameras").json()
    def color_rgb(self, cid):
        d = self.s.get(f"{self.url}/cameras/{cid}/frame/color",
                       params={"format": "jpeg"}).json()
        jpg = base64.b64decode(d['data'])
        bgr = cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)
        return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

cam = CameraClient(); cam.start_all()
cams = cam.list_cameras()
BASE_CID  = cams[0]['serial_number']
WRIST_CID = cams[1]['serial_number'] if len(cams) > 1 else None
print(f"Cameras: base={BASE_CID}  wrist={WRIST_CID}")


In [ ]:
lebai_sdk.init()
lebai = lebai_sdk.connect(ROBOT_IP, False)
lebai.start_sys()    # enable motion (matches grasp_to_the_bowl.py)
lebai.init_claw()    # enable gripper
print(f"Robot connected: {lebai.is_connected()}")


## 4. Build the observation dict

The dict's keys must match exactly what the dataset (and the trained policy) expects. We pull camera frames and current joint + gripper state, normalize to torch tensors, and add a batch dim.

**Image format**: `(1, 3, H, W)`, float in `[0, 1]`. The policy will handle further normalization internally using the `dataset_stats` saved with the checkpoint.

**State format**: `(1, 7)`, float, `[jp0..jp5, claw_amplitude]`. The order *must* match the converter — if you converted with gripper as the 7th dim, you read it as the 7th dim here.


In [ ]:
BASE_IMAGE_KEY  = "observation.images.base"
WRIST_IMAGE_KEY = "observation.images.wrist"

# The policy was trained with a specific feature schema; verify our keys match.
expected_keys = set(policy.config.input_features.keys())
print(f"Policy expects: {expected_keys}")
have_wrist = WRIST_IMAGE_KEY in expected_keys
if WRIST_CID is None and have_wrist:
    raise RuntimeError("Policy was trained with a wrist camera but none is connected.")
if WRIST_CID is not None and not have_wrist:
    print("Note: wrist camera connected but policy doesn't expect it; will be ignored.")


def build_observation():
    """Assemble the dict that policy.select_action() consumes."""
    obs = {}

    # Base camera
    rgb = cam.color_rgb(BASE_CID)                  # (H, W, 3) uint8
    img = torch.from_numpy(rgb).float() / 255.0    # (H, W, 3)
    img = img.permute(2, 0, 1).unsqueeze(0)        # (1, 3, H, W)
    obs[BASE_IMAGE_KEY] = img.to(device, non_blocking=True)

    # Optional wrist camera
    if have_wrist:
        rgb_w = cam.color_rgb(WRIST_CID)
        img_w = torch.from_numpy(rgb_w).float() / 255.0
        img_w = img_w.permute(2, 0, 1).unsqueeze(0)
        obs[WRIST_IMAGE_KEY] = img_w.to(device, non_blocking=True)

    # Proprio: joints + gripper amplitude
    kin = lebai.get_kin_data()
    jp = kin["actual_joint_pose"]
    if isinstance(jp, dict):
        jp = [jp.get(f"jp{i}", jp.get(f"j{i}", 0.0)) for i in range(6)]
    state_list = list(jp[:6])

    if policy.config.input_features["observation.state"].shape[0] == 7:
        try:
            claw = lebai.get_claw()
            amp = float(claw.get("amplitude", 0.0))
        except Exception:
            amp = 0.0
        state_list.append(amp)

    state = torch.tensor(state_list, dtype=torch.float32).unsqueeze(0)
    obs["observation.state"] = state.to(device, non_blocking=True)
    return obs


## 5. Action sender

Two parts:

**Joint command.** Use `lebai.movej(joints, a, v, t, r)` *without* `wait_move()`. The blend radius `r` lets successive commands merge smoothly. `a` and `v` are acceleration and velocity caps; keep them low for first runs.

**Gripper command.** `lebai.set_claw(force, amplitude)`. `amplitude` ranges 0 (closed) to 100 (open) on your firmware (your data collection records it directly). Don't spam this every tick — the gripper is a slow actuator and the policy will produce gripper values that wobble around the true target by a few percent. Send only when the commanded amplitude has changed by more than a small threshold.


In [ ]:
# Tune these for safety. Slow on first runs.
JOINT_ACC_LIMIT = 1.5    # rad/s^2
JOINT_VEL_LIMIT = 1.0    # rad/s
BLEND_RADIUS    = 0.05   # rad — smooth blending between successive movej calls
GRIPPER_FORCE   = 10
GRIPPER_THRESHOLD = 5.0  # only send set_claw if commanded amplitude changed > this

_last_gripper_sent = None

def send_action(action_np):
    """action_np: shape (action_dim,). First 6 dims are joint targets in rad,
    last dim (if 7) is gripper amplitude in [0, 100]."""
    global _last_gripper_sent

    target_joints = [float(x) for x in action_np[:6]]

    # Non-blocking joint command. wait_move() is NOT called — we want
    # the next tick's command to seamlessly blend in.
    lebai.movej(target_joints,
                JOINT_ACC_LIMIT, JOINT_VEL_LIMIT,
                0,                  # 't' arg, 0 means "use a/v limits"
                BLEND_RADIUS)

    # Gripper, rate-limited to avoid spamming.
    if len(action_np) >= 7:
        amp = float(np.clip(action_np[6], 0.0, 100.0))
        if (_last_gripper_sent is None
                or abs(amp - _last_gripper_sent) > GRIPPER_THRESHOLD):
            lebai.set_claw(GRIPPER_FORCE, amp)
            _last_gripper_sent = amp


## 6. Dry run (no robot motion)

Before running for real, do one tick with `send_action` replaced by a `print`. This verifies:

- Camera + state assembly works
- Policy forward pass works
- The action vector looks sane

If anything's off, debug here — not while the robot is moving.


In [ ]:
policy.reset()    # IMPORTANT: clear the temporal-ensembling action queue
obs = build_observation()
with torch.inference_mode():
    action = policy.select_action(obs)
action_np = action[0].cpu().numpy()

print(f"Observation:")
for k, v in obs.items():
    print(f"  {k}: {tuple(v.shape)}  dtype={v.dtype}")
print(f"\nPredicted action ({action_np.shape}):")
print(f"  joint targets (rad): {np.round(action_np[:6], 3)}")
if len(action_np) >= 7:
    print(f"  gripper amplitude:   {action_np[6]:.1f}")

# Compare against current state
current_state = obs['observation.state'][0].cpu().numpy()
print(f"\nCurrent state vs predicted action delta:")
print(f"  joint delta (rad): {np.round(action_np[:6] - current_state[:6], 4)}")


**Read those numbers carefully**

- **Joint targets** should be reasonable for the start of the task (close to current joint values).
- **Joint deltas** should be small, on the order of `velocity * 1/fps` rad. For a slow demonstration at 10 Hz with 0.5 rad/s motion, expect ~0.05 rad. If they're huge (multiple radians), your action representation or normalization is wrong, and running for real will be dangerous.
- **Gripper amplitude** should be roughly in `[0, 100]`. If it's wildly outside that range, normalization is broken.


## 7. The control loop

In [ ]:
PERIOD_S = 0.1                 # 10 Hz
MAX_DURATION_S = 30.0          # safety: hard stop after this long
N_ACTIONS_BEFORE_REQUERY = 1   # 1 = re-query every tick (uses temporal ensembling)
                               # Increase (e.g. to chunk_size//4) if inference is too slow

policy.reset()
print(f"Policy reset. Starting in 3 s. Stop with Jupyter Interrupt or Ctrl-C.")
time.sleep(3)
print("RUNNING.")

t_start = time.time()
next_tick = time.time()
step = 0
_last_gripper_sent = None

try:
    while True:
        loop_t = time.time()
        if loop_t - t_start > MAX_DURATION_S:
            print(f"Reached MAX_DURATION_S={MAX_DURATION_S}s — stopping.")
            break

        obs = build_observation()
        with torch.inference_mode():
            action = policy.select_action(obs)
        action_np = action[0].cpu().numpy()

        send_action(action_np)

        step += 1
        if step % 10 == 0:
            elapsed_loop = time.time() - loop_t
            print(f"  t={loop_t - t_start:5.1f}s  step={step:4d}  "
                  f"loop={elapsed_loop*1000:.0f}ms  "
                  f"j={np.round(action_np[:6], 2)}  "
                  + (f"g={action_np[6]:.0f}" if len(action_np) >= 7 else ""))

        # Steady 10 Hz pacing
        next_tick += PERIOD_S
        sleep_for = next_tick - time.time()
        if sleep_for > 0:
            time.sleep(sleep_for)
        else:
            # Blew the budget. Don't accumulate negative sleep.
            next_tick = time.time()

except KeyboardInterrupt:
    print("\nInterrupted by user.")
finally:
    print(f"Stopped after {step} steps ({time.time() - t_start:.1f}s).")


## 8. Cleanup

In [ ]:
cam.stop_all()
lebai.stop_sys()
print("Camera and robot stopped.")


## 9. Troubleshooting

**The robot moves wildly or in a wrong direction**
- Almost always an action-representation mismatch. Compare the dry-run action against the corresponding action in your *training* data for a similar starting pose. They should look qualitatively similar (same scale, same signs).
- Check joint order. If joint 0 in training corresponds to a different motor than joint 0 at inference, no model in the world will save you.

**The robot doesn't move at all**
- The action might be tiny relative to the joint scale. Check normalization stats in the checkpoint.
- `start_sys()` must be called before any `movej()`. If you forgot, the SDK silently does nothing.
- If `wait_move()` was accidentally added back, the loop is blocked until each move completes — at 10 Hz with default speeds it'll never finish in time.

**The gripper twitches open/close constantly**
- Increase `GRIPPER_THRESHOLD`. The policy outputs a continuous value; threshold turns it back into discrete commands.
- Or convert it to a hard binary in `send_action`: `amp = 100 if action_np[6] > 50 else 0`.

**Choppy joint motion**
- Increase `BLEND_RADIUS`. The default 0.05 rad is conservative; 0.1–0.2 gives smoother arcs but less precise targeting.
- Make sure you're hitting 10 Hz: print `loop_t - prev_loop_t` each tick. If it ever exceeds 100 ms, something in the loop is too slow (usually camera HTTP latency or first-tick CUDA warmup).

**Loss looked great in training but doesn't work in the real world**
- Distribution shift: different lighting, camera position, background, or starting pose vs training. Match training conditions exactly for first runs, then add diversity by collecting more demos and retraining.
- Single-camera policies are particularly weak here. A wrist camera helps a lot because it's tied to the gripper, not the world.


## What to do once it works

1. **Collect demos on the failures.** Whenever the policy fails, reset the scene, demonstrate the recovery, and add those episodes to your dataset. This is the single highest-ROI thing you can do — much higher ROI than tuning hyperparameters.
2. **Add a second task.** Now that the data pipeline is solid, you can compare ACT (which ignores the language label) against a language-conditioned model — you've already got the data for both.
3. **When you're ready for VLAs**, the same dataset feeds π0 / π0.5 fine-tuning via openpi. The data pipeline is identical; only the policy class swaps.
